## Practical work: build a RFM segmentation

### Objectives :
- Apply a RFM (Recency, Frequency, Monetary) analysis to segment customers based on their purchasing behavior.
- Analyze the migrration of customers between segments over 2015 and 2016.

### Import libraies and load data

In [14]:
import numpy as np
import pandas as pd
import seaborn as sns

In [15]:
mypath = "Data"
CUSTOMER = pd.read_csv(mypath +"\\CUSTOMER.csv")
CUSTOMER_ADDITIONAL = pd.read_csv(mypath + "\\CUSTOMER_ADDITIONAL.csv")
PRODUCTS = pd.read_csv(mypath + "\\PRODUCTS.csv")
RECEIPTS = pd.read_csv(mypath + "\\RECEIPTS.csv")
REFERENTIAL = pd.read_csv(mypath + "\\REFERENTIAL.csv", low_memory=False)
STORE = pd.read_csv(mypath + "\\STORE.csv")

## 2 Selection and merging of the data

- Build the CUSTOMERS_INFO table containing all the information about a customer


In [16]:
CUSTOMERS_INFO = pd.merge(CUSTOMER, CUSTOMER_ADDITIONAL, on="ID_INDIVIDU")
CUSTOMERS_INFO.head()

,ID_INDIVIDU,ID_FOYER,CIVILITE,SEXE,PROFESSION,CATEGORIE_PROF,DATE_NAISS_A,DATE_NAISS_M,DATE_NAISS_J,DATE_CREATION_CARTE,CODE_MAGASIN,PAYS,ETAT,TAILLE,TAILLE_SG,TAILLE_BONNET
0,4,175898,0,0,NaN,NaN,NaN,NaN,NaN,01/01/1989,751,FR,N,38.0,95.0,B
1,27,0,3,2,NaN,3.0,1967.0,9.0,1.0,08/08/2007,942,FR,N,38.0,85.0,B
2,34,127060,3,2,NaN,6.0,1953.0,3.0,4.0,27/07/2007,942,FR,N,42.0,95.0,B
3,37,0,2,2,NaN,3.0,1964.0,8.0,21.0,11/07/2007,942,FR,N,42.0,90.0,B
4,47,0,3,2,NaN,NaN,1972.0,3.0,31.0,23/04/2008,941,FR,N,40.0,90.0,C


In [17]:
CUSTOMERS_INFO.rename(columns={"CODE_MAGASIN":"MANAGING_STORE"}, inplace=True)

- Build `RECEIPTS_INFO` merging tables `RECEIPTS` and `REFERENTIAL`:

In [18]:
# We decide to convert the type of `EAN` in `str`!
RECEIPTS['EAN'] = RECEIPTS['EAN'].astype("str")
RECEIPTS_INFO = pd.merge(RECEIPTS, REFERENTIAL, on="EAN", how="left")
RECEIPTS_INFO.head()

,DATE_ACHAT,EAN,ID_INDIVIDU,ID_FOYER,CODE_LIGNE,TYPE_LIGNE,NUM_TICKET,QUANTITE,PRIX_AP_REMISE,REMISE,REMISE_VALEUR,CODE_BOUTIQUE,ID_ARTICLE,ID_MODELE,ID_OPTION,MODELE,OPTION_PTT,COLORIS,POSITION,GRILLE
0,"""14/12/2013""",3585211297939,4,0,81,"""SALE""",29,1,28,0,0,756,41656.0,21.0,1249.0,128,PROME,10.0,2.0,10.0
1,"""14/12/2015""",3585211731150,4,175898,81,"""SALE""",10,1,69,0,0,730,60020.0,116.0,12128.0,222,MERVE,41.0,4.0,3.0
2,"""14/12/2013""",3585210149062,4,0,81,"""SALE""",29,1,0,0,100,756,150.0,7.0,136.0,FAVORI,AGEND,13.0,1.0,5.0
3,"""14/12/2013""",3585211405723,4,0,81,"""SALE""",29,1,23,0,0,756,45761.0,19.0,3056.0,122,SULFU,93.0,1.0,10.0
4,"""02/07/2016""",3585211668319,4,175898,81,"""SALE""",513,1,75,0,0,920,54398.0,136.0,8251.0,503,CAPFE,20.0,4.0,3.0


Add information to the `RECEIPTS` table with `PRODUCTS` and `STORE`:
- Add product information from the `PRODUCTS` table

In [19]:
RECEIPTS_INFO = pd.merge(RECEIPTS_INFO, PRODUCTS[["MODELE", "Ligne", "Famille"]], on="MODELE", how="left")
RECEIPTS_INFO.head()

,DATE_ACHAT,EAN,ID_INDIVIDU,ID_FOYER,CODE_LIGNE,TYPE_LIGNE,NUM_TICKET,QUANTITE,PRIX_AP_REMISE,REMISE,...,ID_ARTICLE,ID_MODELE,ID_OPTION,MODELE,OPTION_PTT,COLORIS,POSITION,GRILLE,Ligne,Famille
0,"""14/12/2013""",3585211297939,4,0,81,"""SALE""",29,1,28,0,...,41656.0,21.0,1249.0,128,PROME,10.0,2.0,10.0,Corseterie,String/Tanga
1,"""14/12/2015""",3585211731150,4,175898,81,"""SALE""",10,1,69,0,...,60020.0,116.0,12128.0,222,MERVE,41.0,4.0,3.0,Homewear,Homewear_Ensemble
2,"""14/12/2013""",3585210149062,4,0,81,"""SALE""",29,1,0,0,...,150.0,7.0,136.0,FAVORI,AGEND,13.0,1.0,5.0,NaN,NaN
3,"""14/12/2013""",3585211405723,4,0,81,"""SALE""",29,1,23,0,...,45761.0,19.0,3056.0,122,SULFU,93.0,1.0,10.0,Corseterie,String/Tanga
4,"""02/07/2016""",3585211668319,4,175898,81,"""SALE""",513,1,75,0,...,54398.0,136.0,8251.0,503,CAPFE,20.0,4.0,3.0,Bain,Bain_Maillot


- Add store information from the `STORE` table

In [20]:
RECEIPTS_INFO = pd.merge(RECEIPTS_INFO, STORE[["CODE_BOUTIQUE", "REGIONS", "CENTRE_VILLE", "TYPE_MAGASIN", "REGIONS_COMMERCIAL"]], on="CODE_BOUTIQUE", how="left")
RECEIPTS_INFO.head()

,DATE_ACHAT,EAN,ID_INDIVIDU,ID_FOYER,CODE_LIGNE,TYPE_LIGNE,NUM_TICKET,QUANTITE,PRIX_AP_REMISE,REMISE,...,OPTION_PTT,COLORIS,POSITION,GRILLE,Ligne,Famille,REGIONS,CENTRE_VILLE,TYPE_MAGASIN,REGIONS_COMMERCIAL
0,"""14/12/2013""",3585211297939,4,0,81,"""SALE""",29,1,28,0,...,PROME,10.0,2.0,10.0,Corseterie,String/Tanga,Paris,Centre ville,Succursale,Paris
1,"""14/12/2015""",3585211731150,4,175898,81,"""SALE""",10,1,69,0,...,MERVE,41.0,4.0,3.0,Homewear,Homewear_Ensemble,Province,Centre ville,Affilié,Province
2,"""14/12/2013""",3585210149062,4,0,81,"""SALE""",29,1,0,0,...,AGEND,13.0,1.0,5.0,NaN,NaN,Paris,Centre ville,Succursale,Paris
3,"""14/12/2013""",3585211405723,4,0,81,"""SALE""",29,1,23,0,...,SULFU,93.0,1.0,10.0,Corseterie,String/Tanga,Paris,Centre ville,Succursale,Paris
4,"""02/07/2016""",3585211668319,4,175898,81,"""SALE""",513,1,75,0,...,CAPFE,20.0,4.0,3.0,Bain,Bain_Maillot,Paris,Centre Commercial,Succursale,Paris


## Data cleaning and feature engineering

In [40]:
# Drop unneeded columns
RECEIPTS_INFO = RECEIPTS_INFO.drop(columns=["REGIONS_COMMERCIAL"])
# Cleaning 'CENTRE_VILLE' column
RECEIPTS_INFO['CENTRE_VILLE'] = RECEIPTS_INFO["CENTRE_VILLE"].replace("Centre Co", "Centre Commercial")
# Drop unneeded columns
RECEIPTS_INFO = RECEIPTS_INFO.drop(columns=["CODE_LIGNE", "TYPE_LIGNE"])
# Drop unneeded columns
RECEIPTS_INFO = RECEIPTS_INFO.drop(columns=["REMISE"])
# Cap REMISE_VALEUR at 100
RECEIPTS_INFO['REMISE_VALEUR'] = RECEIPTS_INFO.loc[RECEIPTS_INFO["REMISE_VALEUR"]>100, "REMISE_VALEUR"]=100
# Feature engineering
RECEIPTS_INFO['PLV'] = np.where(RECEIPTS_INFO["MODELE"]=="PLV", 1, 0)
RECEIPTS_INFO['Gift'] = np.where((RECEIPTS_INFO["MODELE"].isin(["FAVO", "FAVORI"])) & (RECEIPTS_INFO["PRIX_AP_REMISE"]==0),1,0)
RECEIPTS_INFO['Entry-level'] = np.where(RECEIPTS_INFO["MODELE"]=="ACCESS", 1, 0)

### Save the dataframes `RECEIPTS_INFO`:

In [ ]:
RECEIPTS_INFO.to_csv(mypath +"\\RECEIPTS_INFO.csv",index=False)

## Date preparation

- Handle `DATE_ACHAT`

In [45]:
# Create a copy of RECEIPTS_INFO to work on date features
RECEIPTS_INFO_OK = RECEIPTS_INFO.copy()
RECEIPTS_INFO_OK["DATE_ACHAT"] = RECEIPTS_INFO_OK["DATE_ACHAT"].str.replace('"',"",regex=False)
RECEIPTS_INFO_OK["DATE_ACHAT"] = pd.to_datetime(RECEIPTS_INFO_OK["DATE_ACHAT"])

C:\Users\lucas\AppData\Local\Temp\ipykernel_3732\2002087782.py:4: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  RECEIPTS_INFO_OK["DATE_ACHAT"] = pd.to_datetime(RECEIPTS_INFO_OK["DATE_ACHAT"])


- Compute the final price of a purchase with `PRIX_AP_REMISE` x `QUANTITE`.

In [46]:
RECEIPTS_INFO_OK["PRIX"] = RECEIPTS_INFO_OK["PRIX_AP_REMISE"] * RECEIPTS_INFO_OK["QUANTITE"]

### Build the RFM:
#### Create TWO new dfs:
- `RECEIPTS_INFO_RFM_2016`: only keep purchases made in  season 2016 (from september 1, 2015 to august 31, 2016)
- `RECEIPTS_INFO_RFM_2015`: only keep purchases made in 2015 (season 2015: from september 1, 2014 to august 31, 2015)

In [48]:
min_date_RFM_2016 = "2015-08-31"
max_date_RFM_2016 = pd.to_datetime("2016-08-31")

min_date_RFM_2015 = RECEIPTS_INFO_OK['DATE_ACHAT'].max() - pd.DateOffset(days=365*2)
max_date_RFM_2015 = RECEIPTS_INFO_OK['DATE_ACHAT'].max() - pd.DateOffset(days=365)